# Simplified self-attention

This notebook introduces the core attention calculation using six small input
vectors. It follows the simplified example in Chapter 3 of *Build a Large
Language Model (From Scratch)* by Sebastian Raschka.

We will compute pairwise scores, normalize them into attention weights, and
combine the input vectors into one context vector per token. The calculation
processes a single sequence, without a batch dimension.

There are **no trainable projection matrices** in this example. The same input
vectors serve as queries, keys, and values: $Q = K = V = X$.
The attention weights are computed from the input; they are not stored model
parameters. In a larger model, gradients could still flow through these
operations to earlier trainable layers. Here the inputs are fixed illustrative
vectors, so no training takes place.

In [1]:
import torch

## 1. Represent the input sequence

Each row represents one token in “Your journey starts with one step”. The values
are illustrative embeddings, supplied directly rather than obtained from a
trained model. The phrase labels identify the positions; they do not imply that
a real tokenizer always maps each word to exactly one token.

The input shape is `(num_tokens, embedding_dim) = (6, 3)`.
These input vectors are not yet contextualized by this attention calculation.

In [2]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your
        [0.55, 0.87, 0.66],  # journey
        [0.57, 0.85, 0.64],  # starts
        [0.22, 0.58, 0.33],  # with
        [0.77, 0.25, 0.10],  # one
        [0.05, 0.80, 0.55],  # step
    ]
)

print("Input shape:", tuple(inputs.shape))

Input shape: (6, 3)


## 2. Compute a score for every pair of tokens

$$S = XX^T, \qquad S_{ij} = x_i \cdot x_j$$

Row `i` selects a query token; column `j` selects a key token. Their dot product
measures compatibility in this example. Dot products depend on both vector
direction and magnitude; they are not normalized cosine similarities.

The matrix product computes all pairwise scores at once. Multiplying `(6, 3)`
by `(3, 6)` produces a score matrix of shape `(6, 6)`. Because queries and keys
are identical here, the score matrix is symmetric. Learned, different query and
key projections do not generally produce symmetric scores.

We intentionally omit the division by `sqrt(key_dim)` at this introductory stage.
The trainable implementation in `attention.ipynb` introduces scaled dot-product
attention. Adding scaling here would change the weights and outputs.

In [3]:
# .T transposes the two-dimensional input matrix: (6, 3) becomes (3, 6).
attn_scores = inputs @ inputs.T
print("Attention scores (rows: queries, columns: keys):")
print(attn_scores)

Attention scores (rows: queries, columns: keys):
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


## 3. Turn scores into attention weights

$$A_{ij} = \frac{\exp(S_{ij})}{\sum_k \exp(S_{ik})}$$

Softmax is applied along the **last axis**, across all keys for each query.
Each row becomes a distribution of nonnegative weights that sums to one.
A higher score receives a higher weight within that row.

The weights retain shape `(6, 6)`. Although the score matrix is symmetric,
the weight matrix need not be: each row has its own normalization denominator.
Use `torch.softmax` rather than manually exponentiating scores, because it
handles the normalization in a numerically stable way.

In [4]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print("Attention weights:")
print(attn_weights)
print("Row sums:", attn_weights.sum(dim=-1))

Attention weights:
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 4. Combine values into context vectors

$$Z = AX, \qquad z_i = \sum_j A_{ij}x_j$$

In this simplified version, the values are the original input vectors.
For each query, attention combines all values using that query's weights.
The result has shape `(num_tokens, embedding_dim) = (6, 3)`:
one context vector per input token, rather than a single vector for the sentence.

Each output depends on the entire sequence. Here, “contextualized” means that
information from other positions has been mixed in; it does not mean that the
random illustrative embeddings already encode useful learned relationships.

In [5]:
# (6, 6) @ (6, 3) -> (6, 3): each row is a weighted sum of value vectors.
outputs = attn_weights @ inputs
print("Context vector shape:", tuple(outputs.shape))
print(outputs)

Context vector shape: (6, 3)
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 5. Inspect one token's weighted sum

For the second token, “journey” (index `1`), the second row of attention weights
specifies how much each of the six value vectors contributes. Multiplying those
weights into the values and summing must reproduce the second output row.

In [6]:
query_index = 1
# Add a length-one axis so each scalar weight multiplies a full value vector.
weighted_values = attn_weights[query_index].unsqueeze(-1) * inputs
context_vector = weighted_values.sum(dim=0)

print("Weights for 'journey':", attn_weights[query_index])
print("Context vector for 'journey':", context_vector)
print(
    "Matches the matrix calculation:",
    torch.allclose(context_vector, outputs[query_index]),
)

Weights for 'journey': tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Context vector for 'journey': tensor([0.4419, 0.6515, 0.5683])
Matches the matrix calculation: True


## Shapes and limits of this example

| Tensor | Shape | Role |
| --- | --- | --- |
| Inputs, queries, keys, values | `(num_tokens, embedding_dim)` | The same matrix in this simplified version |
| Attention scores | `(num_tokens, num_tokens)` | One score per query–key pair |
| Attention weights | `(num_tokens, num_tokens)` | One distribution over values per query |
| Output context vectors | `(num_tokens, embedding_dim)` | One weighted sum per token |

All tokens can attend to all positions, including themselves and future tokens.
This is **unmasked** attention: a GPT next-token predictor needs a causal mask
to prevent access to future input tokens.

The calculation also has no positional encoding. Reordering the input rows
reorders the output rows in the same way; the mechanism alone does not know
which token is first or last.

In `attention.ipynb`, separate learned projections produce queries, keys, and
values, and scores are divided by the square root of the key dimension. Those
changes make the representations trainable and introduce scaling, while keeping
the same score → softmax → weighted-sum structure.